# Exercise 1 — SMA Crossover

The SMA crossover is the oldest systematic trading rule. When the fast moving average rises above the slow moving average, prices are trending up — go long. When the fast falls below the slow, the trend has reversed — go flat. Simple, interpretable, and effective enough to still be widely used.

In [ ]:
import pandas as pd, math, warnings

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def _sma(s, w):  return s.rolling(w).mean()
def _ema(s, w):  return s.ewm(span=w, adjust=False).mean()
def _rsi(s, w):
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))

def sma_crossover(df, fast=20, slow=50):
    """Long (1) when fast SMA > slow SMA; flat (0) otherwise.

    Steps:
      1. close    = df["Close"]
      2. fast_sma = _sma(close, fast)
      3. slow_sma = _sma(close, slow)
      4. return (fast_sma > slow_sma).fillna(False).astype(int)

    Args:
        df   : OHLCV DataFrame
        fast : fast SMA window (default 20)
        slow : slow SMA window (default 50)
    """
    # TODO: implement the 4 steps above
    return pd.Series(0, index=df.index)


### Checks

In [ ]:
checks = 0

# 1 — returns same-length Series
try:
    df  = _synthetic()
    sig = sma_crossover(df)
    assert isinstance(sig, pd.Series) and len(sig) == len(df)
    checks += 1; print("✅ 1 sma_crossover returns same-length Series")
except Exception as e:
    print("❌ 1:", e)

# 2 — no NaN
try:
    sig = sma_crossover(_synthetic())
    assert not sig.isna().any(), "signal has NaN values"
    checks += 1; print("✅ 2 no NaN values")
except Exception as e:
    print("❌ 2:", e)

# 3 — values are only 0 and 1
try:
    sig = sma_crossover(_synthetic())
    assert set(sig.unique()).issubset({0, 1}), f"unexpected values: {set(sig.unique())}"
    checks += 1; print("✅ 3 signal values are only 0 and 1")
except Exception as e:
    print("❌ 3:", e)

# 4 — produces both 0s and 1s on sine-wave data
try:
    sig = sma_crossover(_synthetic(), fast=10, slow=20)
    assert (sig == 1).any(), "no 1s found — fast never exceeded slow"
    assert (sig == 0).any(), "no 0s found — always long?"
    checks += 1; print("✅ 4 both 0s and 1s present")
except Exception as e:
    print("❌ 4:", e)

# 5 — signal is 1 exactly when fast SMA > slow SMA
try:
    df      = _synthetic()
    close   = df["Close"]
    fast20  = close.rolling(20).mean()
    slow50  = close.rolling(50).mean()
    expected = (fast20 > slow50).fillna(False).astype(int)
    sig      = sma_crossover(df, 20, 50)
    assert (sig == expected).all(), "signal does not match fast_sma > slow_sma"
    checks += 1; print("✅ 5 signal matches fast_sma > slow_sma exactly")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
